In [0]:
%sql
-- Base: depositsplus through 10/14, then deposits_src (with 10/14 enrichment) for newer dates
CREATE OR REPLACE TEMP VIEW topdep_base AS
WITH attrs AS ( -- each account's last known enrichment (10/14), one row per account
SELECT account_number, mgmt_reporting_lvl_3, relationship_name_id, rate_class, y15_class, subproduct_id
FROM (
SELECT regexp_replace(CAST(account_number AS STRING), '-[0-9]+$', '') AS account_number,
mgmt_reporting_lvl_3, relationship_name_id, rate_class, y15_class, subproduct_id,
ROW_NUMBER() OVER (
PARTITION BY regexp_replace(CAST(account_number AS STRING), '-[0-9]+$', '')
ORDER BY ledger_balance DESC
) AS rn
FROM finance.reporting.depositsplus
WHERE asofdate = '2026-10-14'
)
WHERE rn = 1
),
src AS ( -- newer days from deposits_src, with a key that matches depositsplus
SELECT *,
CASE WHEN legacy_bank = 'PNFP' AND account_type IS NOT NULL
THEN substr(accountid, length(account_type) + 1)
ELSE accountid END AS acct_key
FROM finance.staging.deposits_src
WHERE asofdate > '2026-10-14'
),
combined AS (
SELECT asofdate, mgmt_reporting_lvl_3, relationship_name_id, account_name, client_id,
CAST(account_number AS STRING) AS account_number, ledger_balance, interest_rate,
rate_class, product_type, deposit_type, subproduct_id, y15_class
FROM finance.reporting.depositsplus
WHERE asofdate > '2025-12-31'
UNION ALL
SELECT s.asofdate, a.mgmt_reporting_lvl_3, a.relationship_name_id, s.account_name, s.client_id,
s.acct_key, s.ledger_balance, s.interest_rate,
a.rate_class, s.product_type, s.deposit_type, a.subproduct_id, a.y15_class
FROM src s
LEFT JOIN attrs a ON s.acct_key = a.account_number
)
SELECT c.*,
CASE
WHEN mgmt_reporting_lvl_3 IN ('National Capital', 'North Carolina - Virginia Central',
'North Carolina West - South Carolina') THEN 'Carolinas - Callicutt'
WHEN mgmt_reporting_lvl_3 = 'South Florida' THEN 'South Florida - Walker'
WHEN mgmt_reporting_lvl_3 = 'Alabama - Panhandle' THEN 'Alabama - Abele'
WHEN mgmt_reporting_lvl_3 = 'Georgia' THEN 'Georgia - Clark'
WHEN mgmt_reporting_lvl_3 = 'Greater Florida' THEN 'Greater Florida - Keith'
WHEN mgmt_reporting_lvl_3 = 'Tennessee - Kentucky' THEN 'TN/KY - Bean'
END AS Geography
FROM combined c
WHERE mgmt_reporting_lvl_3 IN ('Alabama - Panhandle', 'Georgia', 'Greater Florida', 'National Capital',
'North Carolina - Virginia Central', 'North Carolina West - South Carolina',
'South Florida', 'Tennessee - Kentucky')
AND COALESCE(y15_class, '') NOT LIKE '%Brokered%'
AND COALESCE(deposit_type, '') <> 'Brokered';


In [0]:
# %sql
# -- Base: 2026-09-30 deposits for in-scope regions, mapped to Geography
# CREATE OR REPLACE TEMP VIEW topdep_base AS
# SELECT
#   d.*,
#   CASE
#     WHEN d.mgmt_reporting_lvl_3 IN (
#       'National Capital',
#       'North Carolina - Virginia Central',
#       'North Carolina West - South Carolina'
#     ) THEN 'Carolinas - Callicutt'
#     WHEN d.mgmt_reporting_lvl_3 = 'South Florida'        THEN 'South Florida - Walker'
#     WHEN d.mgmt_reporting_lvl_3 = 'Alabama - Panhandle'  THEN 'Alabama - Abele'
#     WHEN d.mgmt_reporting_lvl_3 = 'Georgia'              THEN 'Georgia - Clark'
#     WHEN d.mgmt_reporting_lvl_3 = 'Greater Florida'      THEN 'Greater Florida - Keith'
#     WHEN d.mgmt_reporting_lvl_3 = 'Tennessee - Kentucky' THEN 'TN/KY - Bean'
#   END AS Geography
# FROM finance.reporting.depositsplus d
# WHERE d.asofdate > '2025-12-31'
#   AND d.mgmt_reporting_lvl_3 IN (
#     'Alabama - Panhandle',
#     'Georgia',
#     'Greater Florida',
#     'National Capital',
#     'North Carolina - Virginia Central',
#     'North Carolina West - South Carolina',
#     'South Florida',
#     'Tennessee - Kentucky'
#   )
#   and coalesce(d.y15_class, '') not LIKE '%Brokered%'
#   and coalesce(d.deposit_type, '') <>'Brokered' ;

In [0]:
%sql
CREATE OR REPLACE TABLE finance.liquidity.core_deposits AS
SELECT
  asofdate,
  Geography,
  TRIM(substring_index(Geography, ' - ', -1)) AS fa_name,
  mgmt_reporting_lvl_3,
  relationship_name_id,
  account_name,
  client_id,
  CAST(account_number AS STRING) AS account_number,
  ledger_balance,
  interest_rate,
  rate_class,
  product_type,
  deposit_type,
  subproduct_id AS product_id
FROM topdep_base;


In [0]:
%sql
CREATE OR REPLACE TABLE finance.liquidity.core_deposits_1m AS
SELECT * FROM finance.liquidity.core_deposits
WHERE ledger_balance >= 1000000;


In [0]:
%sql
select count (*) as rows, count(distinct asofdate) as days
from finance.liquidity.core_deposits_1m

In [0]:
%sql
CREATE OR REPLACE TABLE finance.liquidity.core_deposits_daily AS
SELECT asofdate, Geography, fa_name, mgmt_reporting_lvl_3,
rate_class, product_type, deposit_type,
COUNT(*) AS accounts,
SUM(ledger_balance) AS ledger_balance,
SUM(COALESCE(interest_rate, 0) * ledger_balance) AS rate_x_bal
FROM finance.liquidity.core_deposits
GROUP BY ALL;

select count (*) from finance.liquidity.core_deposits_daily


In [0]:
%sql
SELECT CASE WHEN mgmt_reporting_lvl_3 IN ('Alabama - Panhandle','Georgia','Greater Florida','National Capital',
            'North Carolina - Virginia Central','North Carolina West - South Carolina','South Florida','Tennessee - Kentucky')
            THEN 'In 8 regions' ELSE 'Other regions' END AS scope,
       ROUND(SUM(ledger_balance) / 1e9, 1) AS bal_bn
FROM finance.reporting.depositsplus
WHERE asofdate = '2026-10-14'
  AND COALESCE(y15_class, '') NOT LIKE '%Brokered%'
  AND COALESCE(deposit_type, '') <> 'Brokered'
GROUP BY 1



In [0]:
%sql
SELECT asofdate,
       ROUND(SUM(ledger_balance) / 1e9, 1) AS bal_bn,
       ROUND(SUM(rate_x_bal) / SUM(ledger_balance) * 100, 2) AS wtd_rate_pct
FROM finance.liquidity.core_deposits_daily
WHERE asofdate >= '2026-10-08'
GROUP BY asofdate
ORDER BY asofdate


In [0]:
%sql
-- Relationship totals and balance-weighted average rate per relationship_name_id within each Geography
CREATE OR REPLACE TEMP VIEW topdep_client_totals AS
SELECT
  Geography,
  relationship_name_id,
  COUNT(DISTINCT client_id) AS client_count,
  SUM(currface) AS relationship_total_currface,
  SUM(COALESCE(interest_rate, 0) * currface)
    / NULLIF(SUM(currface), 0) AS wtd_avg_rate_by_relationship
FROM topdep_base
WHERE Geography IS NOT NULL
  AND relationship_name_id IS NOT NULL
GROUP BY Geography, relationship_name_id;

In [0]:
%sql
-- Rank relationships by total balance within each Geography
CREATE OR REPLACE TEMP VIEW topdep_ranked AS
SELECT
  *,
  ROW_NUMBER() OVER (
    PARTITION BY Geography
    ORDER BY relationship_total_currface DESC, relationship_name_id
  ) AS rn
FROM topdep_client_totals;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW topdep_top25 AS
SELECT
  Geography,
  relationship_name_id,
  client_count,
  relationship_total_currface,
  wtd_avg_rate_by_relationship,
  rn
FROM topdep_ranked
WHERE rn <= 25;

In [0]:
%sql
-- Account-level detail for the top 25 relationships
CREATE OR REPLACE TEMP VIEW topdep_top25_accounts AS
SELECT
  b.*,
  t.relationship_total_currface,
  t.wtd_avg_rate_by_relationship,
  t.rn AS relationship_rank
FROM topdep_base b
INNER JOIN topdep_top25 t
  ON  b.Geography = t.Geography
  AND b.relationship_name_id = t.relationship_name_id;

In [0]:
%sql
-- Final output: account detail with Geography-level weighted rate and rate bucket
CREATE OR REPLACE TABLE finance.liquidity.large_deposits AS
SELECT
  a.asofdate,
  a.Geography,
  a.client_id,
  a.relationship_name_id,
  a.relationship_total_currface,
  a.account_number,
  a.account_name,
  a.product_type,
  a.deposit_type,
  a.rate_class,
  a.interest_rate,
  a.currface,
  a.ledger_balance,
  a.maturity_bucket,
  a.days_to_maturity,
  a.account_maturity_date,
  a.rel_insured,
  a.rel_uninsured,
  a.fully_insured_flag,
  a.ops_flag,
  a.fse_flag,
  a.wholesale_flag,
  a.mgmt_reporting_lvl_3,
  a.wtd_avg_rate_by_relationship,
  SUM(COALESCE(a.interest_rate, 0) * a.currface) OVER (PARTITION BY a.Geography)
    / NULLIF(SUM(a.currface) OVER (PARTITION BY a.Geography), 0) AS wtd_avg_rate_by_geography,
  CASE
    WHEN COALESCE(a.interest_rate, 0) = 0                      THEN '0%'
    WHEN a.interest_rate > 0     AND a.interest_rate < 0.02    THEN '0.01%-1.99%'
    WHEN a.interest_rate >= 0.02 AND a.interest_rate < 0.03    THEN '2.00%-2.99%'
    WHEN a.interest_rate >= 0.03 AND a.interest_rate < 0.035   THEN '3.00%-3.49%'
    WHEN a.interest_rate >= 0.035 AND a.interest_rate < 0.04   THEN '3.50%-3.99%'
    WHEN a.interest_rate >= 0.04                               THEN '4.00%+'
  END AS rate_bucket,
  a.relationship_rank
FROM topdep_top25_accounts a;

In [0]:
%sql
SELECT * EXCEPT (relationship_rank)
FROM finance.liquidity.large_deposits

In [0]:
%sql
SELECT * EXCEPT (relationship_rank)
FROM finance.liquidity.large_deposits
ORDER BY Geography, relationship_rank, currface DESC, account_number;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Top 25% percentile version. Depends on topdep_base and topdep_client_totals (cells 1-2).
-- Percentile of each relationship_name_id by total balance within each Geography (largest first)
CREATE OR REPLACE TEMP VIEW topdep_pctl_ranked AS
SELECT
  *,
  ROW_NUMBER() OVER (
    PARTITION BY Geography
    ORDER BY relationship_total_currface DESC, relationship_name_id
  ) AS rn,
  -- Share of relationships in the Geography with a balance >= this one (0.01 = top 1%)
  CUME_DIST() OVER (
    PARTITION BY Geography
    ORDER BY relationship_total_currface DESC
  ) AS balance_pctl_from_top,
  COUNT(*) OVER (PARTITION BY Geography) AS relationships_in_geography
FROM topdep_client_totals;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW topdep_top25pct AS
SELECT
  Geography,
  relationship_name_id,
  client_count,
  relationship_total_currface,
  wtd_avg_rate_by_relationship,
  rn,
  balance_pctl_from_top,
  relationships_in_geography
FROM topdep_pctl_ranked
WHERE balance_pctl_from_top <= 0.25;

In [0]:
%sql
-- Account-level detail for the top 25% relationships
CREATE OR REPLACE TEMP VIEW topdep_top25pct_accounts AS
SELECT
  b.*,
  t.relationship_total_currface,
  t.wtd_avg_rate_by_relationship,
  t.rn AS relationship_rank,
  t.balance_pctl_from_top,
  t.relationships_in_geography
FROM topdep_base b
INNER JOIN topdep_top25pct t
  ON  b.Geography = t.Geography
  AND b.relationship_name_id = t.relationship_name_id;

In [0]:
%sql
-- Final output: account detail for top 25% relationships with Geography-level weighted rate and rate bucket
CREATE OR REPLACE TABLE finance.liquidity.large_deposits_top25pct AS
SELECT
  a.asofdate,
  a.Geography,
  a.client_id,
  a.relationship_name_id,
  a.relationship_total_currface,
  a.balance_pctl_from_top,
  a.relationships_in_geography,
  a.account_number,
  a.account_name,
  a.product_type,
  a.deposit_type,
  a.rate_class,
  a.interest_rate,
  a.currface,
  a.ledger_balance,
  a.maturity_bucket,
  a.days_to_maturity,
  a.account_maturity_date,
  a.rel_insured,
  a.rel_uninsured,
  a.fully_insured_flag,
  a.ops_flag,
  a.fse_flag,
  a.wholesale_flag,
  a.mgmt_reporting_lvl_3,
  a.wtd_avg_rate_by_relationship,
  SUM(COALESCE(a.interest_rate, 0) * a.currface) OVER (PARTITION BY a.Geography)
    / NULLIF(SUM(a.currface) OVER (PARTITION BY a.Geography), 0) AS wtd_avg_rate_by_geography,
  CASE
    WHEN COALESCE(a.interest_rate, 0) = 0                      THEN '0%'
    WHEN a.interest_rate > 0     AND a.interest_rate < 0.02    THEN '0.01%-1.99%'
    WHEN a.interest_rate >= 0.02 AND a.interest_rate < 0.03    THEN '2.00%-2.99%'
    WHEN a.interest_rate >= 0.03 AND a.interest_rate < 0.035   THEN '3.00%-3.49%'
    WHEN a.interest_rate >= 0.035 AND a.interest_rate < 0.04   THEN '3.50%-3.99%'
    WHEN a.interest_rate >= 0.04                               THEN '4.00%+'
  END AS rate_bucket,
  a.relationship_rank
FROM topdep_top25pct_accounts a;

In [0]:
%sql
SELECT * EXCEPT (relationship_rank)
FROM finance.liquidity.large_deposits_top25pct
ORDER BY Geography, relationship_rank, currface DESC, account_number;

In [0]:
%sql
select
d.asofdate,
d.mgmt_reporting_lvl_3,
d.relationship_name_id,
d.client_id,
cast(d.account_number as string) as account_number,
d.ledger_balance,
d.interest_rate,
d.rate_class,
d.product_type,
d.subproduct_id as product_id
from finance.reporting.depositsplus d
where d.asofdate >= add_months (current_date(), -12)
and d.bdsp_flag = 'N' --whhichever column marks brokered




In [0]:
%sql
describe finance.reporting.depositsplus

In [0]:
# %sql



# WITH base AS (
#  SELECT
#  d.*,
# CASE
# WHEN d.mgmt_reporting_lvl_3 IN (
#  'National Capital',
# 'North Carolina - Virginia Central',
#  'North Carolina West - South Carolina'
#  ) THEN 'Carolinas - Callicutt'

#  WHEN d.mgmt_reporting_lvl_3 = 'South Florida'
#  THEN 'South Florida - Walker'

#  WHEN d.mgmt_reporting_lvl_3 = 'Alabama - Panhandle'
#   THEN 'Alabama - Abele'

#  WHEN d.mgmt_reporting_lvl_3 = 'Georgia'
# THEN 'Georgia - Clark'

#  WHEN d.mgmt_reporting_lvl_3 = 'Greater Florida'
# THEN 'Greater Florida - Keith'

#  WHEN d.mgmt_reporting_lvl_3 = 'Tennessee - Kentucky'
#  THEN 'TN/KY - Bean'
#  END AS Geography

# FROM finance.reporting.depositsplus d

# WHERE d.asofdate = '2026-09-30'

#  AND d.mgmt_reporting_lvl_3 IN (
#  'Alabama - Panhandle',
#  'Georgia',
#  'Greater Florida',
#  'National Capital',
#  'North Carolina - Virginia Central',
#  'North Carolina West - South Carolina',
#  'South Florida',
#  'Tennessee - Kentucky'
#  )
# ),

# client_totals AS (
#  SELECT
#  Geography,
#  client_id,

#  -- Display label only, not part of the grouping key
#  MAX(relationship_name_id) AS relationship_name_id,

#  SUM(currface) AS relationship_total_currface,

#  SUM(COALESCE(interest_rate, 0) * currface)
#  / NULLIF(SUM(currface), 0)
#  AS wtd_avg_rate_by_relationship

# FROM base

# WHERE Geography IS NOT NULL
# AND client_id IS NOT NULL

#  GROUP BY
#  Geography,
#  client_id
# ),

# ranked AS (
#  SELECT
#  *,

#  ROW_NUMBER() OVER (
#  PARTITION BY Geography
#  ORDER BY
#  relationship_total_currface DESC,
#  client_id
#   ) AS rn

# FROM client_totals
# ),

# top25 AS (
#  SELECT
# Geography,
#  client_id,
#  relationship_name_id,
#  relationship_total_currface,
#  wtd_avg_rate_by_relationship,
#  rn

#  FROM ranked

# WHERE rn <= 25
# ),

# top25_accounts AS (
#  SELECT
# b.*,
#  t.relationship_total_currface,
# t.wtd_avg_rate_by_relationship,
#  t.rn AS relationship_rank

#  FROM base b

#  INNER JOIN top25 t
#  ON b.Geography = t.Geography
#  AND b.client_id = t.client_id
# )

# SELECT
# a.Geography,
# a.client_id,
# a.relationship_name_id,
# a.relationship_total_currface,
# a.account_number,
# a.account_name,
# a.product_type,
# a.deposit_type,
# a.rate_class,
# a.interest_rate,
# a.currface,
# a.ledger_balance,
# a.maturity_bucket,
# a.days_to_maturity,
# a.account_maturity_date,
# a.rel_insured,
# a.rel_uninsured,
# a.fully_insured_flag,
#  a.ops_flag,
# a.fse_flag,
# a.wholesale_flag,
#  a.mgmt_reporting_lvl_1,
# a.mgmt_reporting_lvl_2,
# a.mgmt_reporting_lvl_3,

# a.wtd_avg_rate_by_relationship,

# SUM(COALESCE(a.interest_rate, 0) * a.currface)
# OVER (PARTITION BY a.Geography)
# / NULLIF(
# SUM(a.currface) OVER (PARTITION BY a.Geography),
# 0
# ) AS wtd_avg_rate_by_geography,

# CASE
# WHEN COALESCE(a.interest_rate, 0) = 0
# THEN '0%'

# WHEN a.interest_rate > 0
# AND a.interest_rate < 0.02
# THEN '0.01%-1.99%'

# WHEN a.interest_rate >= 0.02
# AND a.interest_rate < 0.03
# THEN '2.00%-2.99%'

# WHEN a.interest_rate >= 0.03
# AND a.interest_rate < 0.035
# THEN '3.00%-3.49%'

# WHEN a.interest_rate >= 0.035
# AND a.interest_rate < 0.04
# THEN '3.50%-3.99%'

# WHEN a.interest_rate >= 0.04
# THEN '4.00%+'
# END AS rate_bucket

# FROM top25_accounts a

# ORDER BY
# a.Geography,
# a.relationship_rank,
# a.currface DESC,
# a.account_number;

In [0]:
%sql
-- Percentile ranking of each relationship (client_id) by total balance within each mgmt_reporting_lvl_3
WITH rel AS (
  SELECT
    d.mgmt_reporting_lvl_3,
    d.client_id,
    MAX(d.relationship_name_id) AS relationship_name_id,
    COUNT(*)                    AS account_count,
    SUM(d.currface)             AS relationship_total_currface
  FROM finance.reporting.depositsplus d
  WHERE d.asofdate > '2025-12-31'
    AND d.client_id IS NOT NULL
    AND d.mgmt_reporting_lvl_3 IS NOT NULL
  GROUP BY d.mgmt_reporting_lvl_3, d.client_id
)
SELECT
  mgmt_reporting_lvl_3,
  client_id,
  relationship_name_id,
  account_count,
  relationship_total_currface,
  -- 1.0 = largest relationship in the region, 0.0 = smallest
  ROUND(PERCENT_RANK() OVER (PARTITION BY mgmt_reporting_lvl_3 ORDER BY relationship_total_currface), 4) AS balance_percentile,
  -- Percentile bucket 1-100 (100 = top 1%)
  NTILE(100) OVER (PARTITION BY mgmt_reporting_lvl_3 ORDER BY relationship_total_currface) AS balance_percentile_bucket,
  RANK() OVER (PARTITION BY mgmt_reporting_lvl_3 ORDER BY relationship_total_currface DESC) AS balance_rank,
  COUNT(*) OVER (PARTITION BY mgmt_reporting_lvl_3) AS relationships_in_region
FROM rel
ORDER BY mgmt_reporting_lvl_3, relationship_total_currface DESC

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# In-scope regions only
in_scope_lvl3 = [
    "Alabama - Panhandle",
    "Georgia",
    "Greater Florida",
    "National Capital",
    "North Carolina - Virginia Central",
    "North Carolina West - South Carolina",
    "South Florida",
    "Tennessee - Kentucky",
]

# Relationship (client_id) balances per mgmt_reporting_lvl_3 for 2026-09-30
rel = (
    spark.table("finance.reporting.depositsplus")
    .where((F.col("asofdate") == "2026-09-30")
           & F.col("client_id").isNotNull()
           & F.col("mgmt_reporting_lvl_3").isin(in_scope_lvl3))
           & F.col("account_open)").isin(in_scope_lvl3))
    .groupBy("mgmt_reporting_lvl_3", "client_id")
    .agg(F.sum("currface").cast("double").alias("bal"))

# Percentile band per relationship within its lvl3 (based on balance rank)
w = Window.partitionBy("mgmt_reporting_lvl_3").orderBy(F.col("bal").desc())
band_order = ["Top 1%", "1-5%", "5-10%", "10-25%", "25-50%", "Bottom 50%"]
rel_b = rel.withColumn("pr", F.percent_rank().over(w)).withColumn(
    "band",
    F.when(F.col("pr") < 0.01, "Top 1%")
     .when(F.col("pr") < 0.05, "1-5%")
     .when(F.col("pr") < 0.10, "5-10%")
     .when(F.col("pr") < 0.25, "10-25%")
     .when(F.col("pr") < 0.50, "25-50%")
     .otherwise("Bottom 50%"),
)

# Summary: counts, totals, and balance thresholds at key percentiles
pcts = [0.50, 0.75, 0.90, 0.95, 0.99]
pct_labels = ["P50", "P75", "P90", "P95", "P99"]
summary = (
    rel.groupBy("mgmt_reporting_lvl_3")
    .agg(F.count("*").alias("relationships"),
         F.sum("bal").alias("total_balance"),
         F.percentile_approx("bal", pcts, 10000).alias("p"),
         F.max("bal").alias("max_balance"))
    .select("mgmt_reporting_lvl_3", "relationships", "total_balance",
            *[F.col("p")[i].alias(l) for i, l in enumerate(pct_labels)], "max_balance")
    .orderBy(F.col("total_balance").desc())
    .toPandas()
)

bands = (
    rel_b.groupBy("mgmt_reporting_lvl_3", "band")
    .agg(F.count("*").alias("relationships"), F.sum("bal").alias("balance"))
    .toPandas()
)

display(summary)

order = summary["mgmt_reporting_lvl_3"].tolist()
bal_share = (bands.pivot(index="mgmt_reporting_lvl_3", columns="band", values="balance")
             .reindex(index=order, columns=band_order).fillna(0))
bal_share = bal_share.div(bal_share.sum(axis=1), axis=0)

fig, axes = plt.subplots(3, 1, figsize=(14, 18))

# 1. Relationship counts by lvl3
ax = axes[0]
ax.bar(summary["mgmt_reporting_lvl_3"], summary["relationships"], color="#1f4e79")
ax.set_title("Relationship count by mgmt_reporting_lvl_3")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
for i, v in enumerate(summary["relationships"]):
    ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=8)

# 2. Balance thresholds at percentiles (log scale)
ax = axes[1]
for l in pct_labels:
    ax.plot(summary["mgmt_reporting_lvl_3"], summary[l].clip(lower=1), marker="o", label=l)
ax.set_yscale("log")
ax.set_title("Relationship balance at percentile thresholds by lvl3 (log scale)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(title="Percentile")
ax.grid(True, which="major", axis="y", alpha=0.3)

# 3. Share of total balance held by each percentile band
ax = axes[2]
bottom = None
for b in band_order:
    ax.bar(bal_share.index, bal_share[b], bottom=bottom, label=b)
    bottom = bal_share[b] if bottom is None else bottom + bal_share[b]
ax.set_title("Share of lvl3 balance by relationship percentile band")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(title="Band (by balance rank)", bbox_to_anchor=(1.01, 1), loc="upper left")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT * from finance.staging.deposits_src

In [0]:
%sql
-- Rank relationships by total ledger balance, per asofdate:
--   geography_rank = rank within Geography; total_rank = rank across all Geographies
-- Keeps the top 100 relationships in each Geography.
CREATE OR REPLACE TEMP VIEW topdep_top100_ranked AS
WITH rel AS (
  SELECT
    asofdate,
    Geography,
    relationship_name_id,
    COUNT(DISTINCT client_id) AS client_count,
    SUM(ledger_balance)       AS relationship_ledger_balance
  FROM finance.liquidity.core_deposits
  WHERE Geography IS NOT NULL
    AND relationship_name_id IS NOT NULL
  GROUP BY asofdate, Geography, relationship_name_id
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY asofdate, Geography
      ORDER BY relationship_ledger_balance DESC, relationship_name_id
    ) AS geography_rank,
    ROW_NUMBER() OVER (
      PARTITION BY asofdate
      ORDER BY relationship_ledger_balance DESC, relationship_name_id
    ) AS total_rank
  FROM rel
)
SELECT *
FROM ranked
WHERE geography_rank <= 100;

SELECT *
FROM topdep_top100_ranked
WHERE asofdate = (SELECT MAX(asofdate) FROM topdep_top100_ranked)
ORDER BY Geography, geography_rank;

In [0]:
%sql
WITH s AS (
SELECT *,
CASE WHEN legacy_bank = 'PNFP' AND account_type IS NOT NULL
THEN substr(accountid, length(account_type) + 1)
ELSE accountid END AS acct_key
FROM finance.staging.deposits_src
WHERE asofdate = '2026-10-14'
)
SELECT s.legacy_bank,
COUNT(*) AS src_accounts,
COUNT(p.account_number) AS matched,
ROUND(SUM(s.ledger_balance) / 1e9, 1) AS src_bal_bn,
ROUND(SUM(p.ledger_balance) / 1e9, 1) AS plus_bal_bn
FROM s
LEFT JOIN finance.reporting.depositsplus p
ON CAST(p.account_number AS STRING) = s.acct_key
AND p.asofdate = s.asofdate
GROUP BY s.legacy_bank


In [0]:
%sql
WITH s AS (
  SELECT * FROM finance.staging.deposits_src
  WHERE asofdate = '2026-10-14' AND legacy_bank = 'SNV'
),
u AS (
  SELECT s.* FROM s
  LEFT ANTI JOIN finance.reporting.depositsplus p
    ON CAST(p.account_number AS STRING) = s.accountid
   AND p.asofdate = s.asofdate
)
SELECT acct_typ_desc, product_type, deposit_type, internal_flag,
       COUNT(*) AS accts,
       ROUND(SUM(ledger_balance) / 1e9, 2) AS bal_bn,
       MIN(accountid) AS sample_id
FROM u
GROUP BY ALL
ORDER BY bal_bn DESC
LIMIT 25


In [0]:
%sql
WITH u AS (
  SELECT s.accountid, s.account_name, s.ledger_balance
  FROM finance.staging.deposits_src s
  LEFT ANTI JOIN finance.reporting.depositsplus p
    ON CAST(p.account_number AS STRING) = s.accountid
   AND p.asofdate = s.asofdate
  WHERE s.asofdate = '2026-10-14' AND s.legacy_bank = 'SNV'
    AND s.deposit_type = 'TIME' AND s.ledger_balance > 0
),
p AS (
  SELECT CAST(account_number AS STRING) AS account_number, account_name, ledger_balance
  FROM finance.reporting.depositsplus
  WHERE asofdate = '2026-10-14' AND deposit_type IN ('TIME', 'CDs')
)
SELECT u.accountid AS src_accountid,
       p.account_number AS plus_account_number,
       u.account_name, u.ledger_balance
FROM u
JOIN p
  ON u.account_name = p.account_name
 AND ROUND(u.ledger_balance, 2) = ROUND(p.ledger_balance, 2)
LIMIT 20


In [0]:
%sql
WITH s AS (
SELECT *,
CASE WHEN legacy_bank = 'PNFP' AND account_type IS NOT NULL
THEN substr(accountid, length(account_type) + 1)
ELSE accountid END AS acct_key
FROM finance.staging.deposits_src
WHERE asofdate = '2026-10-14'
),
p AS (
SELECT DISTINCT regexp_replace(CAST(account_number AS STRING), '-[0-9]+$', '') AS acct_key
FROM finance.reporting.depositsplus
WHERE asofdate = '2026-10-14'
)
SELECT s.legacy_bank, s.deposit_type,
COUNT(*) AS src_accounts,
COUNT(p.acct_key) AS matched,
ROUND(SUM(s.ledger_balance) / 1e9, 1) AS src_bal_bn,
ROUND(SUM(CASE WHEN p.acct_key IS NULL THEN s.ledger_balance END) / 1e9, 2) AS unmatched_bal_bn
FROM s
LEFT JOIN p ON s.acct_key = p.acct_key
GROUP BY ALL
ORDER BY legacy_bank, deposit_type


In [0]:
%sql
CREATE OR REPLACE TABLE finance.liquidity.core_deposits_top25 AS
WITH base AS (
SELECT d.*,
CASE
WHEN d.mgmt_lvl_3_desc IN ('National Capital','North Carolina - Virginia Central',
'North Carolina West - South Carolina') THEN 'Carolinas - Callicutt'
WHEN d.mgmt_lvl_3_desc = 'South Florida' THEN 'South Florida - Walker'
WHEN d.mgmt_lvl_3_desc = 'Alabama - Panhandle' THEN 'Alabama - Abele'
WHEN d.mgmt_lvl_3_desc = 'Georgia' THEN 'Georgia - Clark'
WHEN d.mgmt_lvl_3_desc = 'Greater Florida' THEN 'Greater Florida - Keith'
WHEN d.mgmt_lvl_3_desc = 'Tennessee - Kentucky' THEN 'TN/KY - Bean'
END AS Geography
FROM finance.reporting.depositsplus_vw d
WHERE d.asofdate >= '2026-01-01'
),
client_totals AS (
SELECT asofdate, Geography, client_id, SUM(currbook) AS relationship_total
FROM base
WHERE Geography IS NOT NULL AND client_id IS NOT NULL
GROUP BY ALL
),
top25 AS (
SELECT *, ROW_NUMBER() OVER (PARTITION BY asofdate, Geography
ORDER BY relationship_total DESC, client_id) AS relationship_rank
FROM client_totals
QUALIFY relationship_rank <= 25
)
SELECT b.asofdate, b.Geography,
TRIM(substring_index(b.Geography, ' - ', -1)) AS fa_name,
b.mgmt_lvl_3_desc AS mgmt_reporting_lvl_3,
b.relationship_name_id, b.account_name, b.client_id,
b.accountid AS account_number,
b.ledger_balance, b.interest_rate, b.rate_class,
b.product_type, b.deposit_type, b.acct_typ_code AS product_id,
t.relationship_rank
FROM base b
JOIN top25 t
ON b.asofdate = t.asofdate AND b.Geography = t.Geography AND b.client_id = t.client_id;


In [0]:
%sql
SELECT Geography, COUNT(*) AS accts, ROUND(SUM(ledger_balance)) AS bal
FROM finance.liquidity.core_deposits_top25
WHERE asofdate = '2026-09-30'
GROUP BY ROLLUP(Geography) ORDER BY Geography NULLS LAST


In [0]:
%sql
SELECT MAX(asofdate) AS latest_date, COUNT(DISTINCT asofdate) AS days
FROM finance.liquidity.core_deposits_top25
